# CHAT fine-tuning trial (Phase 0 follow-up)

Quick fine-tuning trial: does fine-tuning CHAT's pretrained Kraken recognition model on a
small slice of NomNaOCR's own *training* split close any of the gap found in Phase 0
(CHAT 22.2% vs. NomNaOCR 86.7% on held-out Chu Han characters)? This is a directional signal,
not a rigorous fine-tune - see `experiments/phase0_validation/README.md` in the repo for full
context, and `scripts/build_finetune_data.py` for how the training data (PageXML with
baselines derived from NomNaOCR's bounding boxes) was built.

Training data and CHAT's `chat_rec.mlmodel` are attached as a Kaggle Dataset input.
Only patches from NomNaOCR's `Patches/Train.txt` were used - the held-out validation patches
used for Phase 0's benchmark are never touched here, so the before/after comparison stays honest.

kraken==4.3.13 (pinned for CHAT weight compatibility, per Phase 0's findings) does not build
on Python 3.12 (Kaggle's current default kernel Python) - its packaging predates Python 3.12
and its build backend hits a removed stdlib API (`pkgutil.ImpImporter`). Kaggle's image also
doesn't ship `conda` on PATH, so this notebook installs a self-contained Miniconda under
`/opt/miniconda` (deliberately NOT under `/kaggle/working` - anything there gets treated as
kernel output and re-uploaded/downloaded, which made an earlier attempt at this extremely slow)
and creates a Python 3.10 env there for kraken/ketos, independent of the system Python.

In [ ]:
# kraken 4.3.13 needs Python <=3.11; Kaggle's system Python is 3.12, and Kaggle's image has
# no conda on PATH. Install a self-contained Miniconda + Python 3.10 env under /opt (NOT
# /kaggle/working, which Kaggle treats as kernel output and would otherwise re-upload/
# re-download this whole toolchain on every output fetch).
!curl -sL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/miniconda
CONDA = '/opt/miniconda/bin/conda'
# Newer conda requires explicitly accepting the default channels' Terms of Service before
# `conda create` will use them - without this, env creation fails with CondaToSNonInteractiveError
# and every subsequent `conda run -n kraken_env` silently fails with EnvironmentLocationNotFound.
!{CONDA} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!{CONDA} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!{CONDA} create -y -n kraken_env python=3.10 -q
!{CONDA} run -n kraken_env pip install --quiet kraken==4.3.13
# kraken pulls in an old pytorch_lightning/lightning_fabric that still does
# `pkg_resources.declare_namespace(...)`, a pre-2020 namespace-package pattern. Recent
# setuptools (>=81) dropped the bundled pkg_resources module entirely, so `pip install kraken`
# resolving to a brand-new setuptools breaks that import at ketos-train time with
# "ModuleNotFoundError: No module named 'pkg_resources'". Force a setuptools old enough to
# still ship it.
!{CONDA} run -n kraken_env pip install --quiet "setuptools<81"
# That old pytorch_lightning's RichProgressBar also breaks against modern `rich` (>=13.4
# changed Console.clear_live() in a way that assumes a Live was already pushed, causing
# "IndexError: pop from empty list" the moment training's sanity check starts). Pin an older
# rich that pytorch_lightning's RichProgressBar was actually built against.
!{CONDA} run -n kraken_env pip install --quiet "rich<13.4"
!{CONDA} run -n kraken_env python -c "import torch; print('torch:', torch.__version__); print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"

In [ ]:
import os, glob

def find_dataset_dir(root='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(root):
        if 'chat_rec.mlmodel' in filenames:
            return dirpath
    return None

DATASET_DIR = find_dataset_dir()
print('resolved DATASET_DIR:', DATASET_DIR)
if DATASET_DIR is None:
    print('chat_rec.mlmodel not found anywhere under /kaggle/input - full tree:')
    for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
        print(dirpath, '->', filenames[:10])
    raise FileNotFoundError('chat_rec.mlmodel not found under /kaggle/input')

xml_files = sorted(glob.glob(f'{DATASET_DIR}/finetune_data/*.xml'))
rec_model = f'{DATASET_DIR}/chat_rec.mlmodel'
print('training pages:', len(xml_files))
print('rec model exists:', os.path.exists(rec_model))

In [ ]:
# Quick-trial hyperparameters - adjust EPOCHS if the first run finishes fast and you want to
# push further, or if it's overfitting on this small (~2000 line) subset.
EPOCHS = 10
DEVICE = 'cuda:0'
OUTPUT_PREFIX = '/kaggle/working/chat_finetuned'
CONDA = '/opt/miniconda/bin/conda'

xml_arg = ' '.join(xml_files)
cmd = (
    f"{CONDA} run -n kraken_env ketos train -f page -i {rec_model} --resize union -d {DEVICE} "
    f"-N {EPOCHS} -q dumb -p 0.9 --workers 2 -o {OUTPUT_PREFIX} {xml_arg}"
)
print(cmd[:300], '... (truncated)')
!{cmd}

In [ ]:
import glob, os
checkpoints = sorted(glob.glob('/kaggle/working/chat_finetuned*.mlmodel'))
for c in checkpoints:
    print(c, os.path.getsize(c), 'bytes')